In [78]:
import pandas as pd
import os


In [ ]:
p = "../../results/SpatioTemporalHeteroGNN/eSEEd_v2/RETAIN_50_random_search_summary_ordered_2025-12-30_23-48-52.csv"
param_cols_original = ["kt", "ks", "window_length", "hidden_channels", "use_preprocess_mlp", "add_self_loops"]
param_cols = ["kt", "ks", "window", "hidden_channels", "pre_mlp", "self_loops"]
metric_cols = {"mae": True, "mse": True, "pearson_r": False, "r2": False}

In [80]:
df = pd.read_csv(p).reset_index(drop=True)
# rename window_length to window 
df = df.rename(columns={"window_length": "window"})
df = df.rename(columns={"use_preprocess_mlp": "pre_mlp"})
df = df.rename(columns={"add_self_loops": "self_loops"})
print(df.columns)
print("strategies:", df["strategy"].unique())


Index(['config', 'strategy', 'run_dir', 'timestamp', 'mse', 'mae', 'sd_error',
       'r2', 'pearson_r', 'window', 'kt', 'ks', 'hidden_channels', 'pre_mlp',
       'self_loops'],
      dtype='object')
strategies: ['subject_loo' 'recording_loo']


In [81]:
# for rows, where pre_mlp is False, set hidden_channels to 0
df.loc[df["pre_mlp"] == False, "hidden_channels"] = 0

In [82]:
dfs = df[df["strategy"] == "subject_loo"]
dfr = df[df["strategy"] == "recording_loo"]

In [83]:
# order by metrics and round to 2 decimals for the metric columns
dfs_ordered = {}
dfr_ordered = {}
for metric, ascending in metric_cols.items():
    dfs_ordered[metric] = (
        dfs.sort_values(by=metric, ascending=ascending)
        .assign(**{metric: lambda x: x[metric].round(2)})
    )
    dfr_ordered[metric] = (
        dfr.sort_values(by=metric, ascending=ascending)
        .assign(**{metric: lambda x: x[metric].round(2)})
    )

In [84]:
for metric in metric_cols.keys():
    print(f"Top 10 subject_loo by {metric}:")
    print(dfs_ordered[metric][[metric] + param_cols].head(10))
    print()
    print(f"Top 10 recording_loo by {metric}:")
    print(dfr_ordered[metric][[metric] + param_cols].head(10))
    print()
    print("-----")

Top 10 subject_loo by mae:
     mae  kt  ks  window  hidden_channels  pre_mlp  self_loops
0   2.95   2   1      30                0    False        True
2   2.95  13   1       5                0    False       False
27  2.95  13   3      60              512     True       False
1   2.96   2   6       5                0    False        True
6   2.96   6   3      30              128     True       False
9   2.98   1   1       5               64     True        True
16  2.98  24   4       5              512     True       False
8   2.99  25   5       5               64     True       False
11  2.99  18   3       5                0    False        True
5   2.99   8   5       5               32     True       False

Top 10 recording_loo by mae:
     mae  kt  ks  window  hidden_channels  pre_mlp  self_loops
35  3.06  23   6       5               32     True        True
26  3.07   2   6       5              512     True        True
43  3.09   1   1       5               64     True        Tru